# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 01 p2 dataset audit review

**Permitted partitions: STRUCTURAL AUDIT ARTIFACTS ONLY**

Inspect authoritative station coverage without downloading raw data or repeating the P2 audit.

Protocol: frozen 2026-10-08; provenance uses exact content hashes. Training targets end 2015-02-28; validation ends 2016-02-29. Final test outcomes are not analyzed here.

Prerequisites: open this notebook from the same P4 Git revision you will enter below. In Colab Secrets, set `AIRSENSE_GITHUB_TOKEN` with write access to this repository. Outputs go to `results/<run-id>` on a dedicated Git branch after every checkpoint. Expect setup/download to take minutes; no forecasting model training occurs. If dependency installation requires a runtime restart, restart and rerun setup.


## Runtime bootstrap
Clone a pinned revision in a fresh runtime; preserve existing work on rerun.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = (os.environ.get("AIRSENSE_CODE_REVISION") or input("Code version: full 40-character Git commit SHA (not a run ID): ")).strip().lower()
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-p4.txt"), "-e", str(ROOT)], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)


## Artifact authentication and run identity
The secret is never printed or written into Git configuration. The code SHA selects the software version; the run ID names this notebook's outputs. Use a different run ID for each notebook. Each run pushes compact artifacts to its own `results/<run-id>` branch, keeping source on `main` separate. Reuse the same run ID and original code/environment only for compatible resumes. Updated code requires a new run ID; historical results remain valid under their original revision.

In [ ]:
import logging
from airsense_r.artifacts import provenance, json_bytes
from airsense_r.artifacts import source_hash
from airsense_r.checkpoints import open_run
from airsense_r.colab import configure_github_secret

logging.basicConfig(level=logging.INFO)
if source_hash(ROOT) != "fec10ed8f2a06c91ed372d54799aa04aecd7143dadcc9c28c997ff59b1f486f3":
    raise RuntimeError("Notebook and scientific source revisions differ; open the matching notebook revision")
if TEST_MODE:
    RUN_ID = os.environ["AIRSENSE_NOTEBOOK_TEST_RUN"]
    REMOTE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_REMOTE")
else:
    from google.colab import userdata
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    RUN_ID = input("Run name, e.g. nb03-validation-01 (same name only for resume; not the code SHA): ").strip()
    REMOTE = REPOSITORY


## Evidence and validation
Provenance is checked before restoring outputs. Each completed artifact is saved and pushed before continuing.

In [ ]:
from airsense_r.p4 import review_audit
store = open_run(ROOT, RUN_ID, provenance(ROOT, {"notebook": "01", "partition": "structural-only"}), REMOTE)
summary = review_audit(ROOT)
store.save("audit-review.csv", summary.to_csv(index=False).encode())
display(summary[["station", "rows", "target_missing_pct", "pm25_complete_history_windows"]])


## Interpretation and limitations
Review the displayed evidence before drawing conclusions. Synthetic checks establish implementation behavior, not predictive performance. Training diagnostics cannot justify silently changing frozen settings. Audit missingness is structural evidence. A failed push stops execution with a local checkpoint retained; restart with the same revision/run ID to restore remotely confirmed work. In-flight work not yet uploaded can be lost on runtime termination. See `docs/P4_WORKFLOW.md` for commands and validation limits.